# TinyYOLO Experiment Suite — Notebook 2/7
## COCO val2017 Benchmark (Table IV)

**What:** Train TinyYOLO-q on COCO for the COCO SOTA comparison table.

**GPU Time:** ~24h T4 (3 seeds × ~8h each)

**Platform:** RunPod/Vast.ai recommended. For Kaggle/Colab, run 1 seed per session.

In [ ]:
SEEDS = [42, 123, 256]
EPOCHS = 300
IMGSZ = 416
BATCH = 32
REPO = 'https://github.com/ShMazumder/tinyYOLO.git'

In [ ]:
import os, sys, socket, shutil
from pathlib import Path
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm psutil -q 2>&1 | tail -1

import torch
if torch.cuda.is_available():
    print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')

# Download COCO
print('📁 Downloading COCO (this takes ~10 min)...')
!python3 -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('coco.yaml')" 2>&1 | tail -3
print('✅ Setup complete!')

In [ ]:
import time
for i, seed in enumerate(SEEDS):
    name = f'coco-q-{IMGSZ}-seed{seed}'
    if Path(f'experiments/results/{name}/summary.json').exists():
        print(f'⏭️  {name} — already done'); continue
    print(f'\n{"="*60}\n🏃 [{i+1}/{len(SEEDS)}] {name}\n{"="*60}')
    t0 = time.time()
    !python3 scripts/train.py --task det --variant quantized --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 150 \
        --data coco.yaml --imgsz {IMGSZ} --epochs {EPOCHS} \
        --batch {BATCH} --seed {seed} --warmup 3 \
        --pretrained --compile --name {name}
    print(f'⏱️  {(time.time()-t0)/3600:.1f}h')
print('✅ COCO benchmark complete!')

In [ ]:
import json, glob
import numpy as np
maps = []
for d in sorted(glob.glob(f'experiments/results/coco-q-{IMGSZ}-seed*')):
    s = Path(d) / 'summary.json'
    if s.exists():
        with open(s) as f: data = json.load(f)
        m = data.get('best_map50', 0) * 100
        maps.append(m)
        print(f'  {Path(d).name}: {m:.1f}%')
if maps:
    print(f'\n  TinyYOLO-q COCO mAP@50: {np.mean(maps):.1f} ± {np.std(maps):.1f}%')